# 🟠 PR Score — Automação Loft

**Como usar:**
1. **Célula 1** — Instala dependências (rode uma vez por sessão)
2. **Célula 2** — Configure a chave da API Claude
3. **Célula 3** — Carrega a lista Tier 1
4. **Célula 4** — Upload e limpeza da planilha da Clipadora
5. **Célula 5** — Claude avalia protagonismo matéria por matéria
6. **Célula 6** — Gera planilha .xlsx processada para download
7. **Célula 7** — Gera texto do WhatsApp e HTML do e-mail

---
⚠️ **Pré-requisito:** chave da API Claude em [console.anthropic.com](https://console.anthropic.com) → API Keys

In [ ]:
# ============================================================
# CÉLULA 1 — Instalar dependências
# ============================================================
!pip install anthropic openpyxl requests beautifulsoup4 --quiet

import anthropic
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
import requests
import json
import re
import time
from datetime import date
from bs4 import BeautifulSoup
from google.colab import files
from io import BytesIO

print('✅ Dependências instaladas com sucesso!')

In [ ]:
# ============================================================
# CÉLULA 2 — Configuração
# ============================================================

# Chave da API Claude (obter em console.anthropic.com → API Keys)
ANTHROPIC_API_KEY = ""  # ← Cole sua chave aqui

# Modelo Claude a usar
CLAUDE_MODEL = "claude-sonnet-4-6"

# Data do clipping (padrão: hoje)
DATA_CLIPPING = date.today().strftime("%d/%m/%Y")
# Para outra data, descomente:
# DATA_CLIPPING = "25/06/2026"

assert ANTHROPIC_API_KEY, "❌ Configure o ANTHROPIC_API_KEY!"

# Inicializar cliente
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Teste rápido
teste = client.messages.create(
    model=CLAUDE_MODEL, max_tokens=10,
    messages=[{"role": "user", "content": "Responda apenas: OK"}]
)
print(f"✅ Claude conectado! Resposta: {teste.content[0].text.strip()}")
print(f"✅ Configuração OK — clipping de {DATA_CLIPPING}")

In [ ]:
# ============================================================
# CÉLULA 3 — Carregar lista Tier 1
# ============================================================

print("📋 Faça upload da planilha Tier 1 (Hackaton_Tier_1__Loft_1.xlsx):")
uploaded_tier1 = files.upload()
tier1_filename = list(uploaded_tier1.keys())[0]

df_t1 = pd.read_excel(BytesIO(uploaded_tier1[tier1_filename]), header=None)
TIER1 = set()
for col in df_t1.columns:
    for val in df_t1[col].dropna():
        v = str(val).strip()
        if v:
            TIER1.add(v.lower())

print(f"✅ {len(TIER1)} veículos Tier 1 carregados")

# Constantes
PAYWALL_VEICULOS   = ["valor econômico", "jornal do comércio"]
CANAIS_EXCLUIR     = ["coelho da fonseca", "lopes", "assuntos de interesse"]
TEMAS_BASE = [
    "Mercado Imobiliário", "Precificação", "Locação", "Compra e Venda",
    "Tecnologia e Produto", "Porta-voz", "Pesquisas e Tendências",
    "Expansão e Negócios", "Crise e Reputação", "Mercado de Capitais",
    "Urbanismo e Cidades", "Financiamento e Crédito", "ESG", "RH e Cultura",
    "Transações", "Crédito Imobiliário"
]

# Fills para colorir células
FILL_VERMELHO  = PatternFill(start_color="FFCCCC", end_color="FFCCCC", fill_type="solid")
FILL_AMARELO   = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
FILL_HEADER    = PatternFill(start_color="FF6B35", end_color="FF6B35", fill_type="solid")
FONT_HEADER    = Font(bold=True, color="FFFFFF")

def normalizar_empresa(canal):
    c = str(canal or "").lower()
    if "loft" in c or "foxter" in c:            return "Loft"
    if "zap" in c or "olx" in c or "viva real" in c: return "ZAP"
    if "quintoandar" in c or "quinto andar" in c or "imovelweb" in c: return "QuintoAndar"
    if "superlógica" in c or "superlogica" in c: return "Superlógica"
    if "creditas" in c:                          return "Creditas"
    if "porto seguro" in c:                      return "Porto Seguro"
    return str(canal or "")

def is_tier1(v):    return str(v or "").strip().lower() in TIER1
def is_paywall(v):  return any(p in str(v or "").lower() for p in PAYWALL_VEICULOS)
def is_excluido(c): return any(e in str(c or "").lower() for e in CANAIS_EXCLUIR)

print("✅ Pronto para a Célula 4!")

In [ ]:
# ============================================================
# CÉLULA 4 — Upload e limpeza da planilha da Clipadora
# ============================================================

print("📂 Faça upload da planilha da Clipadora (.xlsx):")
uploaded = files.upload()
clip_filename = list(uploaded.keys())[0]

# Headers na linha 4 (índice 3)
df = pd.read_excel(BytesIO(uploaded[clip_filename]), sheet_name="Matérias", header=3)
total_original = len(df)

# Renomear colunas pelo índice (mais robusto que pelo nome)
# A=0 título, B=1 veículo, C=2 data, G=6 canal, L=11 estado, M=12 mídia, W=22 release, AB=27 link
cols = df.columns.tolist()
def col(i): return cols[i] if i < len(cols) else None

df = df.rename(columns={
    col(0):  "titulo",
    col(1):  "veiculo",
    col(2):  "data_raw",
    col(6):  "canal",
    col(11): "estado",
    col(12): "midia",
    col(26): "url_fonte",   # coluna AA (índice 26)
    col(22): "release",
    col(23): "mat_repetida",
    col(27): "link"
}).copy()

df = df[df["titulo"].notna() & (df["titulo"].astype(str).str.strip() != "")].copy()
print(f"📊 {total_original} matérias no clipping original")

# Corrigir data
def conv_data(v):
    if pd.isna(v): return DATA_CLIPPING
    if isinstance(v, (int, float)):
        try:
            from datetime import datetime, timedelta
            dt = datetime(1899,12,30) + timedelta(days=float(v))
            return dt.strftime("%d/%m/%Y")
        except: pass
    try:
        return pd.to_datetime(v, dayfirst=True).strftime("%d/%m/%Y")
    except:
        return str(v)

df["data"] = df["data_raw"].apply(conv_data)
df["mes"]  = df["data"].apply(lambda d: d.split("/")[1].lstrip("0") if "/" in str(d) else "")
df["empresa"] = df["canal"].apply(normalizar_empresa)

# Motivo para pintar de vermelho
def motivo_vermelho(row):
    if is_excluido(row.get("canal", "")):
        return "Canal excluído"
    if not is_tier1(row.get("veiculo", "")):
        return "Não é Tier 1"
    r = str(row.get("release", "") or "").lower()
    if r in ("sim", "yes", "true", "1"):
        return "Press release pago"
    return ""

df["motivo_vermelho"] = df.apply(motivo_vermelho, axis=1)

n_vermelho = (df["motivo_vermelho"] != "").sum()
df_validas  = df[df["motivo_vermelho"] == ""].reset_index(drop=True)
df_vermelhas = df[df["motivo_vermelho"] != ""]

# Matérias off-topic → Double Check
off_kw = ["carro", "automóvel", "celular", "smartphone", "veículo automotor"]
mask_off = df_validas["titulo"].apply(lambda t: any(k in str(t).lower() for k in off_kw))
df_double_check = df_validas[mask_off].copy()
df_main = df_validas[~mask_off].reset_index(drop=True)

print(f"\n📋 RESUMO DA LIMPEZA")
print(f"   Total original:       {total_original}")
print(f"   Pintadas de vermelho: {n_vermelho}")
print(f"   Para avaliação:       {len(df_main)}")
print(f"   Double Check:         {len(df_double_check)}")

In [ ]:
# ============================================================
# CÉLULA 5 — Avaliação de protagonismo com Claude
# ============================================================

PROMPT = """Você é um especialista em avaliação de PR Score para a Loft (mercado imobiliário brasileiro).

EMPRESA MONITORADA: {empresa}
TÍTULO: {titulo}
VEÍCULO: {veiculo}
CONTEÚDO:
{conteudo}

---
LÓGICA OBRIGATÓRIA — siga esta ordem:

PASSO 1 — Se algum critério C9-C19 for verdadeiro → DESTAQUE.
Se nenhum for verdadeiro → MENÇÃO (exceto se C1 ou C2 forem verdadeiros).

C9:  ≥3 frases de porta-voz da empresa
C10: ≥5 frases ou 8 linhas referenciando a empresa/porta-voz
C11: empresa ocupa ≥30% do espaço total da matéria
C12: live com 1k+ visualizações [→ necessarios_avaliacao_humana]
C13: empresa é fonte de dados usados na matéria (mais comum — se usou dado, pontua)
C14: citação em coluna jornalística (até 300 toques = coluna; acima = texto normal)
C15: veículo nichado estratégico [→ necessarios_avaliacao_humana]
C16: sentenças positivas e negativas alternadas sobre fatos distintos
C17: publieditorial que parece editorial (leitor não identifica como pago)
C18: anúncio do governo (federal/estadual/municipal de capital) com empresa nominalmente
C19: equivalências de marca — OLX/Viva Real→ZAP, ImovelWeb→QuintoAndar, Foxter→Loft

PASSO 2 (só se Destaque) — pontos extras, cada um verdadeiro = +1:
C1:  empresa no título ou linha-fina
C2:  foto com executivo/logo/legenda [→ necessarios_avaliacao_humana se incerto]
C3:  TV/Rádio duração [→ necessarios_avaliacao_humana]
C4:  TV/Rádio aspas de porta-voz [→ necessarios_avaliacao_humana]
C5:  chamada em redes sociais ou home do portal [→ necessarios_avaliacao_humana]
C6:  gráfico/infográfico/box mencionando empresa [→ necessarios_avaliacao_humana se incerto]
C7:  artigo escrito por executivo da empresa
C8:  ≥10 trechos de porta-voz no lead [→ necessarios_avaliacao_humana]
C20: ≥6 menções à empresa ou produtos no texto
C21: CTA explícito ('leia mais', 'veja também') para outra matéria citando a empresa

Tom negativo → "Destaque Negativo" ou "Menção Negativa".
pr_tec_produto e pr_puro_imobis → apenas Loft, senão sempre false.
Dado Proativo (data=true): foco da matéria é o dado/estudo fornecido pela empresa.
Dado de Carona (data_carona=true): dado aparece como apoio em matéria de outro assunto.
Temas disponíveis: {temas}

Responda EXCLUSIVAMENTE com JSON válido, sem markdown, sem texto adicional:
{{"protagonismo":"Destaque|Menção|Destaque Negativo|Menção Negativa","tipo":"Online|Impresso|Rádio|TV|Chamada Online","pr_tec_produto":false,"pr_puro_imobis":false,"data":false,"data_carona":false,"tema":"","subtema":"","produto":"","pontuacao":0,"criterios_considerados":"","necessarios_avaliacao_humana":"","obs":"","confianca":"alta|media|baixa"}}"""


def raspar_texto(url):
    if not str(url or "").startswith("http"): return ""
    try:
        r = requests.get(str(url), headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        for t in soup(["script","style","nav","header","footer"]): t.decompose()
        return " ".join(p.get_text().strip() for p in soup.find_all("p"))[:4500]
    except: return ""


def avaliar(row):
    titulo  = str(row.get("titulo",  ""))
    veiculo = str(row.get("veiculo", ""))
    empresa = str(row.get("empresa", ""))

    if is_paywall(veiculo):
        return {"protagonismo":"Menção","tipo":"Impresso","pr_tec_produto":False,
                "pr_puro_imobis":False,"data":False,"data_carona":False,
                "tema":"","subtema":"","produto":"","pontuacao":0,
                "criterios_considerados":"","necessarios_avaliacao_humana":"PAYWALL",
                "obs":f"🔴 PAYWALL — {veiculo}: acessar com login premium",
                "confianca":"baixa"}

    texto = raspar_texto(row.get("url_fonte","") or row.get("link",""))
    sem_texto = not texto
    conteudo = texto or f"[Texto indisponível — avalie pelo título: {titulo}]"

    prompt = PROMPT.format(
        empresa=empresa, titulo=titulo, veiculo=veiculo,
        conteudo=conteudo, temas=", ".join(TEMAS_BASE)
    )

    for tentativa in range(3):
        try:
            msg = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=800,
                messages=[{"role":"user","content":prompt}]
            )
            raw = msg.content[0].text.strip()
            raw = re.sub(r"```json\s*","",raw); raw = re.sub(r"```\s*","",raw)
            res = json.loads(raw)
            if sem_texto:
                h = res.get("necessarios_avaliacao_humana","")
                res["necessarios_avaliacao_humana"] = ";".join(filter(None,[h,"SEM_TEXTO"]))
                res["confianca"] = "baixa"
            return res
        except json.JSONDecodeError:
            time.sleep(2)
        except Exception as e:
            if tentativa < 2: time.sleep(2**(tentativa+1))
            else:
                return {"protagonismo":"Menção","tipo":"Online","pr_tec_produto":False,
                        "pr_puro_imobis":False,"data":False,"data_carona":False,
                        "tema":"","subtema":"","produto":"","pontuacao":0,
                        "criterios_considerados":"","necessarios_avaliacao_humana":"ERRO",
                        "obs":f"🔴 Erro: {str(e)[:80]}","confianca":"baixa"}


print(f"🤖 Avaliando {len(df_main)} matérias com Claude ({CLAUDE_MODEL})...")
print(f"   Estimativa: ~{max(1, len(df_main)*5//60)}–{max(2, len(df_main)*8//60)} minutos\n")

avaliacoes = []
for i, (_, row) in enumerate(df_main.iterrows()):
    print(f"   [{i+1}/{len(df_main)}] {str(row.get('titulo',''))[:65]}...")
    avaliacoes.append(avaliar(row))
    time.sleep(0.5)

df_av = pd.DataFrame(avaliacoes)
df_main = pd.concat([df_main.reset_index(drop=True), df_av], axis=1)

print(f"\n✅ AVALIAÇÃO CONCLUÍDA")
print(f"   Destaques:    {(df_main['protagonismo']=='Destaque').sum()}")
print(f"   Menções:      {(df_main['protagonismo']=='Menção').sum()}")
print(f"   Negativos:    {df_main['protagonismo'].str.contains('Negativo').sum()}")
print(f"   Revisão 🔴:   {df_main['necessarios_avaliacao_humana'].fillna('').str.strip().ne('').sum()}")

In [ ]:
# ============================================================
# CÉLULA 6 — Gerar planilha .xlsx processada para download
# ============================================================

print("📊 Gerando planilha processada...")

# Ler planilha original com openpyxl para preservar formatação
uploaded_clip_bytes = list(uploaded.values())[0]
wb_orig = openpyxl.load_workbook(BytesIO(uploaded_clip_bytes))
ws_orig = wb_orig["Matérias"] if "Matérias" in wb_orig.sheetnames else wb_orig.active

wb_out = openpyxl.Workbook()
wb_out.remove(wb_out.active)  # remover aba padrão

# ── ABA 1: "Cópia de Matérias" (backup idêntico) ──────────────
ws_copia = wb_out.create_sheet("Cópia de Matérias")
for row in ws_orig.iter_rows():
    for cell in row:
        ws_copia[cell.coordinate] = cell.value
print("   ✅ Aba 'Cópia de Matérias' criada")

# ── ABA 2: "Matérias" com linhas pintadas de vermelho ─────────
ws_mat = wb_out.create_sheet("Matérias")
for row in ws_orig.iter_rows():
    for cell in row:
        ws_mat[cell.coordinate] = cell.value

# Pintar linhas vermelhas
# Os dados começam na linha 5 do arquivo (linha 1=título, 2-3=vazias, 4=header, 5+=dados)
LINHA_INICIO_DADOS = 5
n_colunas = ws_orig.max_column

# Mapear títulos → linha na planilha original
titulos_para_pintar = set(df[df["motivo_vermelho"] != ""]["titulo"].astype(str).str.strip())
linhas_pintadas = 0

for xl_row in ws_mat.iter_rows(min_row=LINHA_INICIO_DADOS):
    titulo_celula = str(xl_row[0].value or "").strip()
    if titulo_celula in titulos_para_pintar:
        for cell in xl_row:
            cell.fill = FILL_VERMELHO
        linhas_pintadas += 1

print(f"   ✅ Aba 'Matérias' criada — {linhas_pintadas} linhas pintadas de vermelho")

# ── ABA 3: "Gabarito Dashboard" ───────────────────────────────
ws_gab = wb_out.create_sheet("Gabarito Dashboard")

HEADERS_GAB = [
    "Índice","Mês","Título","Veículo","Data","Canal","Link","Empresa",
    "Protagonismo","Tipo","PR Tec+Produto","PR Puro Imobis","Data","Data Carona",
    "Retranca","Modelo Clipping","Tema","Subtema","Produto","OBS","Chave Subscore",
    "Cidade","Estado","Pontuação da Matéria","Critérios Considerados na Pontuação",
    "Necessários Avaliação Humana"
]

ws_gab.append(HEADERS_GAB)
for i, cell in enumerate(ws_gab[1], 1):
    cell.fill = FILL_HEADER
    cell.font = FONT_HEADER

for i, (_, row) in enumerate(df_main.iterrows(), 1):
    titulo = str(row.get("titulo",""))
    veiculo = str(row.get("veiculo",""))
    link = str(row.get("link",""))
    linha = [
        i,
        row.get("mes",""),
        titulo,
        veiculo,
        row.get("data", DATA_CLIPPING),
        str(row.get("canal","")),
        link,
        str(row.get("empresa","")),
        str(row.get("protagonismo","Menção")),
        str(row.get("tipo","Online")),
        "TRUE" if row.get("pr_tec_produto") else "FALSE",
        "TRUE" if row.get("pr_puro_imobis") else "FALSE",
        "TRUE" if row.get("data")           else "FALSE",
        "TRUE" if row.get("data_carona")    else "FALSE",
        "",  # Retranca — humano preenche
        f"*[Retranca]* {titulo} - {veiculo} {link}",
        str(row.get("tema","")),
        str(row.get("subtema","")),
        str(row.get("produto","")),
        str(row.get("obs","")),
        f"{i}{veiculo}",  # Chave Subscore
        "",  # Cidade
        str(row.get("estado","")),
        row.get("pontuacao", 0),
        str(row.get("criterios_considerados","")),
        str(row.get("necessarios_avaliacao_humana",""))
    ]
    ws_gab.append(linha)

    # Pintar amarelo se precisar revisão humana
    xl_row_num = i + 1  # +1 por causa do header
    if str(row.get("necessarios_avaliacao_humana","")).strip():
        for c in range(1, len(HEADERS_GAB)+1):
            ws_gab.cell(row=xl_row_num, column=c).fill = FILL_AMARELO

# Ajustar largura das colunas
ws_gab.column_dimensions["C"].width = 60  # Título
ws_gab.column_dimensions["G"].width = 40  # Link
ws_gab.column_dimensions["T"].width = 50  # OBS

print(f"   ✅ Aba 'Gabarito Dashboard' criada — {len(df_main)} linhas")

# ── ABA 4: "Double Check" ─────────────────────────────────────
if len(df_double_check) > 0:
    ws_dc = wb_out.create_sheet("Double Check")
    ws_dc.append(["Título","Veículo","Data","Canal","Empresa","Link","OBS"])
    for _, r in df_double_check.iterrows():
        ws_dc.append([
            str(r.get("titulo","")), str(r.get("veiculo","")),
            str(r.get("data",DATA_CLIPPING)), str(r.get("canal","")),
            str(r.get("empresa","")), str(r.get("link","")),
            "⚠️ Verificar contexto — possível off-topic"
        ])
    print(f"   ✅ Aba 'Double Check' criada — {len(df_double_check)} linhas")

# ── ABA 5: "Totais" ───────────────────────────────────────────
ws_tot = wb_out.create_sheet("Totais")
EMP = ["Loft","QuintoAndar","ZAP","Superlógica"]
pts, dests = {}, {}
for e in EMP: pts[e]=0; dests[e]=[]

for _, r in df_main.iterrows():
    if r.get("protagonismo")=="Destaque" and r.get("empresa") in pts:
        pts[r["empresa"]] += int(r.get("pontuacao") or 1)
        dests[r["empresa"]].append(r)

por_midia  = df_main["tipo"].value_counts().to_dict()
por_estado = df_main["estado"].value_counts().head(10).to_dict()
n_humano   = df_main["necessarios_avaliacao_humana"].fillna("").str.strip().ne("").sum()

ws_tot.append(["PR Score — Totais do Dia"])
ws_tot.append([])
ws_tot.append(["EMPRESA","Pontos","Destaques"])
for e in EMP: ws_tot.append([e, pts[e], len(dests[e])])
ws_tot.append([])
ws_tot.append(["TIPO DE MÍDIA","Qtd"])
for k,v in sorted(por_midia.items(), key=lambda x:-x[1]): ws_tot.append([k,v])
ws_tot.append([])
ws_tot.append(["ESTADO (top 10)","Qtd"])
for k,v in sorted(por_estado.items(), key=lambda x:-x[1]): ws_tot.append([k,v])
ws_tot.append([])
ws_tot.append(["RESUMO",""])
ws_tot.append(["Total avaliadas",  len(df_main)])
ws_tot.append(["Pintadas vermelho", n_vermelho])
ws_tot.append(["Revisão humana",   int(n_humano)])

print("   ✅ Aba 'Totais' criada")

# ── Salvar e baixar ───────────────────────────────────────────
data_arquivo = DATA_CLIPPING.replace("/","-")
nome_arquivo = f"PR_Score_{data_arquivo}.xlsx"
wb_out.save(nome_arquivo)
print(f"\n📥 Iniciando download de '{nome_arquivo}'...")
files.download(nome_arquivo)
print("✅ Planilha gerada com sucesso!")

In [ ]:
# ============================================================
# CÉLULA 7 — Gerar WhatsApp e HTML do e-mail
# ============================================================

CORES = {"Loft":"#FF6B35","QuintoAndar":"#1A73E8","ZAP":"#34A853","Superlógica":"#212121"}
EMOJIS = {"Loft":"🟠","QuintoAndar":"🔵","ZAP":"🟢","Superlógica":"⚫"}

# ── WhatsApp ─────────────────────────────────────────────────
wa = f"📊 PR Score — {DATA_CLIPPING}\n\n"
for e in EMP:
    wa += f"{EMOJIS[e]} {e}: {pts[e]} pt{'s' if pts[e]!=1 else ''}\n"

if dests["Loft"]:
    wa += "\n📰 Destaques Loft do dia:\n"
    for d in dests["Loft"]:
        wa += f"• {d.get('titulo','')} - {d.get('veiculo','')} {d.get('link','')}\n"

if n_humano > 0:
    wa += f"\n⚠️ {int(n_humano)} matéria{'s' if n_humano!=1 else ''} aguardam revisão humana"

print("=" * 60)
print("📱 MENSAGEM WHATSAPP — copie e cole no grupo")
print("=" * 60)
print(wa)

# ── HTML e-mail ───────────────────────────────────────────────
def secao(empresa):
    cor = CORES[empresa]
    ds  = dests.get(empresa, [])
    if not ds:
        itens = '<p style="color:#888;font-style:italic">Sem destaques hoje.</p>'
    else:
        itens = "".join(f"""
      <div style="border-left:3px solid {cor};padding:10px 16px;margin-bottom:10px;background:#fafafa">
        <a href="{d.get('link','#')}" style="color:#111;font-weight:600;font-size:15px;text-decoration:none">{d.get('titulo','')}</a>
        <div style="margin-top:4px;font-size:13px;color:#666">
          {d.get('veiculo','')} · {d.get('data','')}
          <span style="background:{cor};color:#fff;padding:2px 8px;border-radius:12px;font-size:11px;margin-left:6px">{int(d.get('pontuacao',1))} pt{'s' if int(d.get('pontuacao',1))!=1 else ''}</span>
        </div>
      </div>""" for d in ds)
    return f"""
    <div style="margin-bottom:28px">
      <h2 style="color:{cor};border-bottom:2px solid {cor};padding-bottom:6px">
        {EMOJIS[empresa]} {empresa}
        <span style="font-size:14px;font-weight:normal;color:#888"> — {pts[empresa]} pontos</span>
      </h2>
      {itens}
    </div>"""

placar = " ".join(
    f'<div style="text-align:center;padding:10px 16px">'
    f'<div style="font-size:22px;font-weight:700;color:{CORES[e]}">{pts[e]}</div>'
    f'<div style="font-size:12px;color:#666">{e}</div></div>'
    for e in EMP
)

html = f"""<!DOCTYPE html>
<html lang="pt-BR"><head><meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>PR Score — {DATA_CLIPPING}</title></head>
<body style="font-family:Arial,sans-serif;max-width:660px;margin:0 auto;padding:24px;color:#222">
  <div style="background:#FF6B35;padding:20px 24px;border-radius:8px;margin-bottom:20px">
    <h1 style="color:#fff;margin:0;font-size:22px">📊 PR Score</h1>
    <p style="color:#fff;margin:4px 0 0;opacity:.9">{DATA_CLIPPING}</p>
  </div>
  <div style="display:flex;justify-content:space-around;background:#f5f5f5;border-radius:8px;margin-bottom:28px">
    {placar}
  </div>
  {''.join(secao(e) for e in EMP)}
  <div style="border-top:1px solid #eee;padding-top:14px;font-size:12px;color:#999;text-align:center">
    PR Score · Loft · Time de PR
  </div>
</body></html>"""

nome_html = f"email_pr_score_{DATA_CLIPPING.replace('/','-)}.html"
with open(nome_html, "w", encoding="utf-8") as f:
    f.write(html)
files.download(nome_html)

print("\n" + "="*60)
print(f"📧 HTML salvo e baixado: {nome_html}")
print("="*60)
print("\n✅ TUDO PRONTO! Próximos passos:")
print(f"   1. Revisar planilha — linhas vermelhas e linhas amarelas (revisão humana)")
print(f"   2. Copiar Gabarito Dashboard e colar na planilha oficial do Drive")
print(f"   3. Preencher coluna Retranca (Monday)")
print(f"   4. Enviar WhatsApp → aguardar joinha do gestor")
print(f"   5. Após joinha: colar HTML no RD Station e enviar")